In [ ]:
   import torch
   print(torch.__version__)
   print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
   from google.colab import drive
   drive.mount('/content/drive')
   import os
   os.makedirs('/content/drive/MyDrive/ai-detector', exist_ok=True)

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/ai-detector', exist_ok=True)
print("folder ready")

In [ ]:
   import os
   from google.colab import userdata
   os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!pip install -q --upgrade kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip
!ls /content/data

In [ ]:
!ls /content/data/real_vs_fake
!ls /content/data/real_vs_fake/real-vs-fake
!ls /content/data/real_vs_fake/real-vs-fake/train


In [ ]:
!pip install -q datasets
from datasets import load_dataset

ds = load_dataset("julienlucas/midjourney-dalle-sd-nanobananapro-dataset")
print(ds)
print(ds["train"].features)


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# 1. How many of each class?
names = ds["train"].features["label"].names
counts = Counter(ds["train"]["label"])
for i, n in enumerate(names):
    print(n, counts[i])

# 2. Show 8 random images with their labels
import random
idxs = random.sample(range(len(ds["train"])), 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.flat, idxs):
    item = ds["train"][i]
    ax.imshow(item["image"])
    ax.set_title(names[item["label"]])
    ax.axis("off")
plt.show()

# 3. Image sizes and color modes vary, so check a few
for i in idxs[:5]:
    img = ds["train"][i]["image"]
    print(img.size, img.mode)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Training images get a random flip so the model sees slight variety
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
# Validation and test images stay unchanged, for fair scoring
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# A small wrapper that lets PyTorch read our Hugging Face dataset
class HFImages(Dataset):
    def __init__(self, hf_ds, tf):
        self.ds = hf_ds
        self.tf = tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        img = item["image"].convert("RGB")   # fixes mode P / RGBA images
        return self.tf(img), item["label"]

# Split 15% of train off as validation, keeping fake/real balance
split = ds["train"].train_test_split(test_size=0.15, seed=42, stratify_by_column="label")

train_ds = HFImages(split["train"], train_tf)
val_ds   = HFImages(split["test"],  eval_tf)    # this is our validation set
test_ds  = HFImages(ds["test"],     eval_tf)    # untouched until the end

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print("train:", len(train_ds), "| val:", len(val_ds), "| test:", len(test_ds))

x, y = next(iter(train_dl))
print("batch shape:", x.shape)
print("labels:", y[:10])

In [ ]:
import torch.nn as nn
from torchvision import models

# 1. Use the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# 2. Load EfficientNet-B0 with pretrained weights (downloads ~20 MB, free)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# 3. Replace the last layer: originally 1000 classes, now just 2 (fake, real)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

# 4. Loss function: measures how wrong the model is
loss_fn = nn.CrossEntropyLoss()

# 5. Optimizer: adjusts the model to reduce the loss (lr = step size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 6. Test: push one batch through, before any training
x, y = next(iter(train_dl))
out = model(x.to(device))
print("output shape:", out.shape)

In [ ]:
import time, os

# One pass over a dataset. train=True updates the model, train=False only scores it.
def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)                  # 1. guess
            loss = loss_fn(out, y)          # 2. measure how wrong
            if train:
                optimizer.zero_grad()
                loss.backward()             # 3. work out how to improve
                optimizer.step()            # 4. nudge the model
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

EPOCHS = 5
best_acc = 0.0
os.makedirs("/content/drive/MyDrive/ai-detector", exist_ok=True)
save_path = "/content/drive/MyDrive/ai-detector/detector.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl, train=False)
    print(f"Epoch {epoch}/{EPOCHS} | train acc {tr_acc:.3f} | "
          f"val acc {va_acc:.3f} | val loss {va_loss:.3f} | {time.time()-t0:.0f}s")
    if va_acc > best_acc:                    # keep only the best version
        best_acc = va_acc
        torch.save({"state_dict": model.state_dict(),
                    "classes": ["fake", "real"]}, save_path)
        print("  saved best model to Drive")

print("Done. Best val accuracy:", best_acc)

In [ ]:
import numpy as np

# Load the best saved model
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

preds, labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        out = model(x.to(device))
        preds += out.argmax(1).cpu().tolist()
        labels += y.tolist()

p, l = np.array(preds), np.array(labels)
print("Test accuracy:", round((p == l).mean(), 3))
for i, name in enumerate(["fake", "real"]):
    m = l == i
    print(f"{name}: {(p[m] == i).mean():.3f} correct ({m.sum()} images)")
print("fake called real:", ((l == 0) & (p == 1)).sum())
print("real called fake:", ((l == 1) & (p == 0)).sum())

In [ ]:
import shutil, os
shutil.rmtree("/content/my_test", ignore_errors=True)
os.makedirs("/content/my_test/ai")
os.makedirs("/content/my_test/real")
print("ready")

In [ ]:
from google.colab import files
os.chdir("/content/my_test/ai")
files.upload()
os.chdir("/content")


In [ ]:
os.chdir("/content/my_test/real")
files.upload()
os.chdir("/content")

In [ ]:
!ls /content/my_test/ai
!ls /content/my_test/real

In [ ]:
from PIL import Image
from pathlib import Path

model.eval()

def p_ai(path):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    return probs[0].item()          # index 0 = fake

for folder in ["ai", "real"]:
    right = total = 0
    for f in sorted(Path(f"/content/my_test/{folder}").glob("*")):
        if f.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        p = p_ai(f)
        guess = "ai" if p >= 0.5 else "real"
        right += (guess == folder)
        total += 1
        print(f"{folder:5} {f.name[:30]:30} P(ai)={p:.2f} -> {guess}")
    print(f"== {folder}: {right}/{total} correct\n")

In [ ]:
import os, torch, torch.nn as nn
from torchvision import models, transforms
from google.colab import drive

# 1. Reconnect Drive (click through the permission popup if it appears)
drive.mount('/content/drive')

# 2. Rebuild the same model shape and load your saved weights
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

save_path = "/content/drive/MyDrive/ai-detector/detector.pt"
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model = model.to(device).eval()

# 3. The same image preparation we used for validation and testing
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

print("Model loaded. Classes:", ckpt["classes"])

In [ ]:
!ls /content/my_test/ai
!ls /content/my_test/real

In [ ]:
from PIL import Image
from pathlib import Path

model.eval()

def p_ai(path):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    return probs[0].item()          # index 0 = fake

for folder in ["ai", "real"]:
    right = total = 0
    for f in sorted(Path(f"/content/my_test/{folder}").glob("*")):
        if f.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        p = p_ai(f)
        guess = "ai" if p >= 0.5 else "real"
        right += (guess == folder)
        total += 1
        print(f"{folder:5} {f.name[:30]:30} P(ai)={p:.2f} -> {guess}")
    print(f"== {folder}: {right}/{total} correct\n")

In [ ]:
   import os
   from google.colab import files
   os.makedirs("/content/my_test2/real", exist_ok=True)
   os.chdir("/content/my_test2/real")
   files.upload()
   os.chdir("/content")

In [ ]:
   right = total = 0
   for f in sorted(Path("/content/my_test2/real").glob("*")):
       if f.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
           continue
       p = p_ai(f)
       guess = "ai" if p >= 0.5 else "real"
       right += (guess == "real")
       total += 1
       print(f"{f.name[:30]:30} P(ai)={p:.2f} -> {guess}")
   print(f"== real photos: {right}/{total} correct")

In [ ]:
import random, shutil
from pathlib import Path

random.seed(1)
exts = {".jpg", ".jpeg", ".png", ".webp"}
sources = {
    "ai":   ["/content/my_test/ai"],
    "real": ["/content/my_test/real", "/content/my_test2/real"],
}
shutil.rmtree("/content/my_data", ignore_errors=True)

for label, folders in sources.items():
    files = [f for d in folders for f in Path(d).glob("*") if f.suffix.lower() in exts]
    random.shuffle(files)
    half = len(files) // 2
    for i, f in enumerate(files):
        split = "train" if i < half else "test"
        dest = Path(f"/content/my_data/{split}/{label}")
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest / f"{i}_{f.name}")
    print(label, "-> train:", half, "| test:", len(files) - half)

In [ ]:
import io, random, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
exts = {".jpg", ".jpeg", ".png", ".webp"}

# 1. Randomly shrink and/or compress an image, like WhatsApp or a screenshot does
class RandomDegrade:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.5:
            s = random.uniform(0.4, 0.9)
            w, h = img.size
            img = img.resize((max(64, int(w * s)), max(64, int(h * s))))
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(30, 90))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

train_tf = transforms.Compose([
    transforms.Resize(512),
    RandomDegrade(0.5),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# 2. Original dataset (same split as before: seed 42)
ds = load_dataset("julienlucas/midjourney-dalle-sd-nanobananapro-dataset")
split = ds["train"].train_test_split(test_size=0.15, seed=42, stratify_by_column="label")

class HFImages(Dataset):
    def __init__(self, hf_ds, tf):
        self.ds, self.tf = hf_ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        return self.tf(item["image"].convert("RGB")), item["label"]

# 3. Your own images (folder "ai" = label 0 fake, folder "real" = label 1 real)
class FolderImages(Dataset):
    def __init__(self, root, tf):
        self.tf, self.items = tf, []
        for folder, label in [("ai", 0), ("real", 1)]:
            for f in sorted((Path(root) / folder).glob("*")):
                if f.suffix.lower() in exts:
                    self.items.append((f, label))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        f, y = self.items[i]
        return self.tf(Image.open(f).convert("RGB")), y

own_train = FolderImages("/content/my_data/train", train_tf)
own_test  = FolderImages("/content/my_data/test", eval_tf)
print("own train:", len(own_train), "| own hidden test:", len(own_test))

# 4. Mix: original training data + your images repeated 20 times
train_ds = ConcatDataset([HFImages(split["train"], train_tf)] + [own_train] * 20)
val_ds = HFImages(split["test"], eval_tf)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)
own_dl = DataLoader(own_test, batch_size=16)

# 5. Start from the model you already trained
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
ckpt = torch.load("/content/drive/MyDrive/ai-detector/detector.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"])
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)   # smaller steps: we're fine-tuning

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()
    p, l = torch.tensor(preds), torch.tensor(labels)
    acc = (p == l).float().mean().item()
    ai_acc = (p[l == 0] == 0).float().mean().item()
    real_acc = (p[l == 1] == 1).float().mean().item()
    return acc, ai_acc, real_acc

save_v2 = "/content/drive/MyDrive/ai-detector/detector_v2.pt"
best = 0.0
for epoch in range(1, 4):
    model.train()
    correct = n = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = loss_fn(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
    val_acc, _, _ = evaluate(val_dl)
    _, own_ai, own_real = evaluate(own_dl)
    print(f"Epoch {epoch}/3 | train {correct/n:.3f} | dataset val {val_acc:.3f} | "
          f"YOUR hidden test: ai {own_ai:.2f}, real {own_real:.2f}")
    score = (own_ai + own_real) / 2
    if score >= best:
        best = score
        torch.save({"state_dict": model.state_dict(), "classes": ["fake", "real"]}, save_v2)
        print("  saved detector_v2.pt")

In [ ]:
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q /content/val2017.zip -d /content/coco
!ls /content/coco/val2017 | wc -l

In [ ]:
import io, random, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
from datasets import load_dataset
from google.colab import drive

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)
random.seed(0)
exts = {".jpg", ".jpeg", ".png", ".webp"}
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# 1. Randomly shrink/compress images (applies to BOTH classes)
class RandomDegrade:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.5:
            s = random.uniform(0.4, 0.9)
            w, h = img.size
            img = img.resize((max(64, int(w * s)), max(64, int(h * s))))
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(30, 90))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

train_tf = transforms.Compose([
    transforms.Resize(512), RandomDegrade(0.5),
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])

# 2. The original dataset (same split as before)
ds = load_dataset("julienlucas/midjourney-dalle-sd-nanobananapro-dataset")
split = ds["train"].train_test_split(test_size=0.15, seed=42, stratify_by_column="label")

class HFImages(Dataset):
    def __init__(self, hf_ds, tf):
        self.ds, self.tf = hf_ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        return self.tf(item["image"].convert("RGB")), item["label"]

# 3. Any list of (file path, label) -> dataset.  Labels: fake = 0, real = 1
class FileList(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        f, y = self.items[i]
        return self.tf(Image.open(f).convert("RGB")), y

# 4. COCO real photos: 4500 for training, 500 held back for checking
coco = sorted(Path("/content/coco/val2017").glob("*.jpg"))
random.shuffle(coco)
coco_test, coco_train = coco[:500], coco[500:]

# 5. Your own images from before (folders: ai = 0, real = 1)
def own_items(part):
    items = []
    for folder, label in [("ai", 0), ("real", 1)]:
        for f in sorted(Path(f"/content/my_data/{part}/{folder}").glob("*")):
            if f.suffix.lower() in exts:
                items.append((f, label))
    return items
own_tr, own_te = own_items("train"), own_items("test")
print("your images -> train:", len(own_tr), "| hidden test:", len(own_te))

# 6. Combine everything for training (your images repeated 4 times)
train_ds = ConcatDataset(
    [HFImages(split["train"], train_tf),
     FileList([(f, 1) for f in coco_train], train_tf)]
    + [FileList(own_tr, train_tf)] * 4
)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl  = DataLoader(HFImages(split["test"], eval_tf), batch_size=32, num_workers=2)
coco_dl = DataLoader(FileList([(f, 1) for f in coco_test], eval_tf), batch_size=32)
own_dl  = DataLoader(FileList(own_te, eval_tf), batch_size=16)

# 7. Class weights: the rarer class counts more in the loss
hf_labels = split["train"]["label"]
n_fake = hf_labels.count(0) + 4 * sum(1 for _, l in own_tr if l == 0)
n_real = hf_labels.count(1) + len(coco_train) + 4 * sum(1 for _, l in own_tr if l == 1)
total = n_fake + n_real
weights = torch.tensor([total / (2 * n_fake), total / (2 * n_real)]).to(device)
print("fake:", n_fake, "| real:", n_real, "| weights:", weights.tolist())

# 8. Start from the FIRST model you trained
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
ckpt = torch.load("/content/drive/MyDrive/ai-detector/detector.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"])
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

# 9. Accuracy on fake images and on real images, separately
@torch.no_grad()
def scores(loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()
    p, l = torch.tensor(preds), torch.tensor(labels)
    def acc(c):
        m = l == c
        return (p[m] == c).float().mean().item() if m.any() else float("nan")
    return acc(0), acc(1)

# 10. Train 3 epochs, saving a file after each one
for epoch in range(1, 4):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    v_fake, v_real = scores(val_dl)
    _, c_real = scores(coco_dl)
    o_ai, o_real = scores(own_dl)
    print(f"Epoch {epoch} | dataset val: fake {v_fake:.2f} real {v_real:.2f} | "
          f"COCO real {c_real:.2f} | YOUR test: ai {o_ai:.2f} real {o_real:.2f}")
    torch.save({"state_dict": model.state_dict(), "classes": ["fake", "real"]},
               f"/content/drive/MyDrive/ai-detector/detector_v3_e{epoch}.pt")

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh /content/drive/MyDrive/ai-detector

In [ ]:
!pip install -q datasets

In [ ]:
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q /content/val2017.zip -d /content/coco
!ls /content/coco/val2017 | wc -l

In [ ]:
coco_test, coco_train = coco[:500], coco[500:]

In [ ]:
import io, random, time, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
from datasets import load_dataset
from google.colab import drive

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)
random.seed(0)
exts = {".jpg", ".jpeg", ".png", ".webp"}
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# 1. Randomly shrink/compress images (applies to BOTH classes)
class RandomDegrade:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        if random.random() < 0.5:
            s = random.uniform(0.4, 0.9)
            w, h = img.size
            img = img.resize((max(64, int(w * s)), max(64, int(h * s))))
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(30, 90))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

train_tf = transforms.Compose([
    transforms.Resize(512), RandomDegrade(0.5),
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])

# 2. The original dataset (same split as before)
ds = load_dataset("julienlucas/midjourney-dalle-sd-nanobananapro-dataset")
split = ds["train"].train_test_split(test_size=0.15, seed=42, stratify_by_column="label")

class HFImages(Dataset):
    def __init__(self, hf_ds, tf):
        self.ds, self.tf = hf_ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        return self.tf(item["image"].convert("RGB")), item["label"]

# 3. Any list of (file path, label) -> dataset.  Labels: fake = 0, real = 1
class FileList(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        f, y = self.items[i]
        return self.tf(Image.open(f).convert("RGB")), y

# 4. COCO real photos: 2000 for training, 500 held back for checking
coco = sorted(Path("/content/coco/val2017").glob("*.jpg"))
random.shuffle(coco)
coco_test, coco_train = coco[:500], coco[500:2500]

# 5. Your own images (gone after the reset, so these lists will be empty)
def own_items(part):
    items = []
    for folder, label in [("ai", 0), ("real", 1)]:
        for f in sorted(Path(f"/content/my_data/{part}/{folder}").glob("*")):
            if f.suffix.lower() in exts:
                items.append((f, label))
    return items
own_tr, own_te = own_items("train"), own_items("test")
print("your images -> train:", len(own_tr), "| hidden test:", len(own_te))

# 6. Smaller mix so it finishes fast
hf_small = split["train"].select(random.sample(range(len(split["train"])), 3500))
train_ds = ConcatDataset(
    [HFImages(hf_small, train_tf),
     FileList([(f, 1) for f in coco_train], train_tf)]
    + [FileList(own_tr, train_tf)] * 4
)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl  = DataLoader(HFImages(split["test"].select(range(600)), eval_tf),
                     batch_size=32, num_workers=2)
coco_dl = DataLoader(FileList([(f, 1) for f in coco_test], eval_tf), batch_size=32)
own_dl  = DataLoader(FileList(own_te, eval_tf), batch_size=16)

# 7. Class weights: the rarer class counts more in the loss
hf_labels = hf_small["label"]
n_fake = hf_labels.count(0) + 4 * sum(1 for _, l in own_tr if l == 0)
n_real = hf_labels.count(1) + len(coco_train) + 4 * sum(1 for _, l in own_tr if l == 1)
total = n_fake + n_real
weights = torch.tensor([total / (2 * n_fake), total / (2 * n_real)]).to(device)
print("fake:", n_fake, "| real:", n_real, "| weights:", weights.tolist())

# 8. Start from the FIRST model you trained
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
ckpt = torch.load("/content/drive/MyDrive/ai-detector/detector.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"])
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

# 9. Accuracy on fake images and on real images, separately
@torch.no_grad()
def scores(loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()
    p, l = torch.tensor(preds), torch.tensor(labels)
    def acc(c):
        m = l == c
        return (p[m] == c).float().mean().item() if m.any() else float("nan")
    return acc(0), acc(1)

# 10. Train 3 epochs, printing speed, saving a file after each one
for epoch in range(1, 4):
    model.train()
    t0 = time.time()
    for i, (x, y) in enumerate(train_dl):
        if i % 25 == 0:
            print(f"epoch {epoch} batch {i}/{len(train_dl)}  {time.time()-t0:.0f}s")
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    v_fake, v_real = scores(val_dl)
    _, c_real = scores(coco_dl)
    print(f"Epoch {epoch} | dataset val: fake {v_fake:.2f} real {v_real:.2f} | "
          f"COCO real {c_real:.2f}")
    torch.save({"state_dict": model.state_dict(), "classes": ["fake", "real"]},
               f"/content/drive/MyDrive/ai-detector/detector_v3_e{epoch}.pt")

In [ ]:
!ls -lh /content/drive/MyDrive/ai-detector